# GeoValora AI — Model Explainability with SHAP

Public portfolio notebook derived from the interpretability stage of **GeoValora AI**.

This version is intentionally sanitized for public sharing:

- no original or processed dataset is included;
- no row-level TEST predictions are exposed;
- no Google Drive or private filesystem paths are used;
- no serialized model or SHAP explainer is redistributed;
- only aggregated explainability results and selected visual evidence are consumed.

The complete academic workflow remains in the private TFM repository.

## Objective

The interpretability stage answers three questions:

1. Which variables contribute most strongly to model predictions?
2. Is the global feature ranking consistent with an independent importance method?
3. Are the explanations reasonably stable across Madrid, Barcelona and Valencia?

The original academic analysis used the frozen **XGBoost** model with a **52-variable** feature contract.

## Method summary

The private academic notebook computed SHAP values from a reproducible city-balanced sample and compared them with permutation importance.

Selected aggregate results:

| Diagnostic | Result |
|---|---:|
| Model variables | 52 |
| Balanced SHAP sample | 2,100 observations |
| SHAP observations per city | 700 |
| Permutation-importance sample | 3,000 observations |
| Spearman rank correlation: SHAP vs permutation | 0.9625 |
| Common variables in both Top 10 rankings | 8 / 10 |
| Minimum SHAP rank correlation across cities | 0.9683 |
| Minimum bootstrap rank correlation | 0.9987 |

These diagnostics support the stability of the global interpretation while avoiding a causal interpretation of SHAP values.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")

SHAP_RESULTS = RESULTS_DIR / "importancia_shap_global.csv"
SHAP_IMAGE = IMAGES_DIR / "shap_summary.png"

print("Aggregated SHAP results available:", SHAP_RESULTS.exists())
print("SHAP summary image available:", SHAP_IMAGE.exists())

## Aggregated global importance

The public repository contains only the aggregated SHAP importance table exported by the application package.

The following cell inspects the available columns and builds a compact ranking without requiring access to the original observations.

In [ ]:
shap_global = pd.read_csv(SHAP_RESULTS)

print("Rows:", len(shap_global))
print("Columns:", list(shap_global.columns))

shap_global.head()

In [ ]:
label_candidates = [
    "Etiqueta_usuario",
    "Variable",
    "feature",
    "Feature",
]

value_candidates = [
    "SHAP_ABS_MEDIO",
    "mean_abs_shap",
    "Mean_ABS_SHAP",
    "importance",
    "Importance",
]

label_col = next((c for c in label_candidates if c in shap_global.columns), None)
value_col = next((c for c in value_candidates if c in shap_global.columns), None)

if label_col is None or value_col is None:
    raise ValueError(
        "Could not identify the feature-label and SHAP-importance columns. "
        f"Available columns: {list(shap_global.columns)}"
    )

top_shap = (
    shap_global[[label_col, value_col]]
    .dropna()
    .sort_values(value_col, ascending=False)
    .head(15)
)

top_shap

In [ ]:
plot_data = top_shap.sort_values(value_col)

ax = plot_data.plot(
    kind="barh",
    x=label_col,
    y=value_col,
    legend=False,
    figsize=(9, 6),
)

ax.set_title("Top global SHAP feature importance")
ax.set_xlabel("Mean absolute SHAP value")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## SHAP direction summary

The original analysis also used a SHAP summary plot to show both **magnitude** and **direction** of feature contributions.

High and low feature values can push an individual prediction in different directions. This describes the behavior of the fitted model; it does **not** demonstrate that changing a variable would causally change a property's market price.

In [ ]:
if SHAP_IMAGE.exists():
    display(Image(filename=str(SHAP_IMAGE)))
else:
    print("Add images/shap_summary.png to display the portfolio visualization.")

## Interpretation

The private analysis found that the model relies strongly on **geographical** and **structural** information.

Among the most influential variables were:

- approximate latitude;
- constructed area;
- distance to the urban axis;
- distance to the city centre;
- corrected number of bathrooms;
- amenity count;
- cadastral quality.

The original notebook also checked coordinate importance through a dedicated ablation analysis and tested explanation stability across cities and bootstrap resamples.

## Responsible interpretation

SHAP values answer:

> *How did the model distribute this prediction across its input variables?*

They do **not** answer:

> *What would happen in the real market if one characteristic were changed?*

For that reason, GeoValora AI treats SHAP as an explainability mechanism rather than a causal or investment recommendation tool.

## Public-data boundary

This portfolio notebook intentionally excludes:

- original Idealista18 records;
- cleaned or modeled row-level datasets;
- row-level TEST predictions;
- representative property IDs and local cases;
- the serialized final model;
- the serialized SHAP explainer;
- private project paths and Google Drive dependencies.

The goal is to demonstrate the analytical approach while respecting the project's redistribution policy.